In [3]:
!cat ../../third_party/ragdoll-codegen/compiler/src/iree/compiler/Codegen/LLVMGPU/test/set_transform_strategy_batch_matmul.mlir

// RUN: iree-opt %s --split-input-file --pass-pipeline="builtin.module(hal.executable(hal.executable.variant(iree-llvmgpu-select-lowering-strategy)))" \
// RUN:     --iree-codegen-llvmgpu-enable-transform-dialect-jit=1 --iree-codegen-llvmgpu-enable-transform-dialect-batch-matmul-strategy |\
// RUN:   FileCheck %s --check-prefixes=CHECK,DEFAULT

// RUN: iree-opt %s --split-input-file --pass-pipeline="builtin.module(hal.executable(hal.executable.variant(iree-llvmgpu-select-lowering-strategy)))" \
// RUN:     --iree-codegen-llvmgpu-enable-transform-dialect-jit=1 --iree-codegen-llvmgpu-enable-transform-dialect-batch-matmul-strategy \
// RUN: -td-matmul-strategy-blk-sizes=128,64,32,2 \
// RUN: -td-matmul-strategy-reduc-size=8 \
// RUN: -td-matmul-strategy-num-threads=32,4,1 \
// RUN: -td-matmul-strategy-num-warps=1,4,1 \
// RUN: -td-matmul-strategy-use-async-copies=true \
// RUN: -td-matmul-strategy-pipeline-depth=3 \
// RUN: -td-matmul-strategy-use-mma-sync=false \
// RUN: -td-matmul-strat

In [4]:
!iree-opt ../../third_party/ragdoll-codegen/compiler/src/iree/compiler/Codegen/LLVMGPU/test/set_transform_strategy_batch_matmul.mlir \
--pass-pipeline="builtin.module(hal.executable(hal.executable.variant(iree-llvmgpu-select-lowering-strategy)))" \
--iree-codegen-llvmgpu-enable-transform-dialect-jit=1 --iree-codegen-llvmgpu-enable-transform-dialect-batch-matmul-strategy > ../../build/.td_test_0

In [40]:
!iree-opt ../../third_party/ragdoll-codegen/compiler/src/iree/compiler/Codegen/LLVMGPU/test/set_transform_strategy_batch_matmul.mlir \
--pass-pipeline="builtin.module(hal.executable(hal.executable.variant(iree-llvmgpu-select-lowering-strategy)))" \
--td-matmul-strategy-blk-sizes=128,64,32,2 \
--td-matmul-strategy-reduc-size=4 \
--td-matmul-strategy-num-threads=32,4,1 \
--td-matmul-strategy-num-warps=1,4,1 \
--td-matmul-strategy-use-async-copies=true \
--td-matmul-strategy-pipeline-depth=2 \
--td-matmul-strategy-use-mma-sync=false \
--td-matmul-strategy-use-fma=true \
--iree-codegen-llvmgpu-enable-transform-dialect-jit=1 --iree-codegen-llvmgpu-enable-transform-dialect-batch-matmul-strategy > ../../build/.td_test_1
!cat ../../build/.td_test_1

#executable_target_cuda_nvptx_fb = #hal.executable.target<"cuda", "cuda-nvptx-fb", {target_arch = "sm_80"}>
#map = affine_map<(d0, d1, d2, d3) -> (d0, d1, d3)>
#map1 = affine_map<(d0, d1, d2, d3) -> (d0, d3, d2)>
#map2 = affine_map<(d0, d1, d2, d3) -> (d0, d1, d2)>
#pipeline_layout = #hal.pipeline.layout<push_constants = 0, sets = [<0, bindings = [<0, storage_buffer, ReadOnly>, <1, storage_buffer, ReadOnly>, <2, storage_buffer>]>]>
#translation = #iree_codegen.translation_info<TransformDialectCodegen>
#device_target_cuda = #hal.device.target<"cuda", {executable_targets = [#executable_target_cuda_nvptx_fb], legacy_sync}>
module attributes {hal.device.targets = [#device_target_cuda]} {
  hal.executable private @batch_matmul_dispatch_0 {
    hal.executable.variant public @cuda_nvptx_fb target(#executable_target_cuda_nvptx_fb) {
      hal.executable.export public @batch_matmul_dispatch_0_generic_128x80x320x32_f32 ordinal(0) layout(#pipeline_layout) attributes {translation_info = #translati

In [41]:
!diff ../../build/.td_test_0 ../../build/.td_test_1

40c40
<             %tiled_op, %forall_op = transform.structured.tile_using_forall %0#1 tile_sizes [64, 64, 1](mapping = [#gpu.block<z>, #gpu.block<y>, #gpu.block<x>]) : (!transform.any_op) -> (!transform.any_op, !transform.any_op)
---
>             %tiled_op, %forall_op = transform.structured.tile_using_forall %0#1 tile_sizes [128, 64, 32](mapping = [#gpu.block<z>, #gpu.block<y>, #gpu.block<x>]) : (!transform.any_op) -> (!transform.any_op, !transform.any_op)
52c52
<             %tiled_linalg_op, %loops = transform.structured.tile_using_for %tiled_op[0, 0, 0, 16] : (!transform.any_op) -> (!transform.any_op, !transform.any_op)
---
>             %tiled_linalg_op, %loops = transform.structured.tile_using_for %tiled_op[0, 0, 0, 4] : (!transform.any_op) -> (!transform.any_op, !transform.any_op)
83c83
<             %tiled_op_0, %forall_op_1 = transform.structured.tile_using_forall %10 num_threads [1, 32, 4](mapping = [#gpu.thread<linear_dim_2>, #gpu.thread<linear_dim_1>, #gpu.thread<linear_d

In [44]:
!iree-compile ../../build/.td_test_0 \
-o ../../build/.td_test.vmfb \
--compile-from=executable-configurations \
--iree-hal-target-backends=cuda \
--iree-hal-cuda-llvm-target-arch=sm_70

../../build/.td_test_0:17:9: error: 'func.func' op uses 282624 bytes of shared memory; exceeded the limit of 166912 bytes
        func.func @batch_matmul_dispatch_0_generic_128x80x320x32_f32() {
        ^
../../build/.td_test_0:17:9: note: see current operation: 
"func.func"() <{function_type = () -> (), sym_name = "batch_matmul_dispatch_0_generic_128x80x320x32_f32"}> ({
  %0 = "arith.constant"() <{value = dense<0.000000e+00> : vector<1xf32>}> : () -> vector<1xf32>
  %1 = "arith.constant"() <{value = dense<0.000000e+00> : vector<4xf32>}> : () -> vector<4xf32>
  %2 = "arith.constant"() <{value = 63 : index}> : () -> index
  %3 = "arith.constant"() <{value = 62 : index}> : () -> index
  %4 = "arith.constant"() <{value = 61 : index}> : () -> index
  %5 = "arith.constant"() <{value = 60 : index}> : () -> index
  %6 = "arith.constant"() <{value = 59 : index}> : () -> index
  %7 = "arith.constant"() <{value = 58 : index}> : () -> index
  %8 = "arith.constant"() <{value = 57 : index}> : () ->